In [ ]:
# ========= 0. INSTALL =========
!pip install -U qiskit qiskit-aer qiskit-machine-learning scikit-learn xgboost joblib --quiet

# ========= 1. IMPORTS =========
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

from qiskit_machine_learning.algorithms import VQC
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.optimizers import COBYLA

# ========= 2. CONFIG =========
SEED = 42
CONFIG = {"test_size": 0.2}

# ========= 3. LOAD DATA =========
df = pd.read_csv("/content/500_records.csv")
df.columns = df.columns.str.strip().str.lower()

print("Columns:", df.columns.tolist())

# ========= 4. TARGET FIX =========
possible_targets = ['status', 'placement_status', 'placed']
TARGET = next((c for c in possible_targets if c in df.columns), None)

if TARGET is None:
    raise Exception(" Target column not found")

df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()

mapping = {
    "placed": 1, "yes": 1, "1": 1, "true": 1,
    "not placed": 0, "no": 0, "0": 0, "false": 0
}

y = df[TARGET].map(mapping)

# Drop invalid rows
mask = y.notna()
df = df.loc[mask]
y = y.loc[mask].astype(int)

X = df.drop(columns=[TARGET])

if X.shape[0] == 0:
    raise Exception(" Dataset empty after cleaning")

print("Final dataset size:", X.shape)

# ========= 5. FEATURE TYPES =========
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# ========= 6. CLASS WEIGHTS =========
classes = np.unique(y)
if len(classes) > 1:
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    class_weights = dict(zip(classes, weights))
else:
    class_weights = None

# ========= 7. PIPELINE =========
def build_pipeline(model):

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', MinMaxScaler())
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

    return Pipeline([
        ('preprocess', preprocessor),
        ('feature_select', SelectKBest(score_func=f_classif, k='all')),
        ('model', model)
    ])

# ========= 8. MODELS =========
models = {
    "RF": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=SEED),

    "XGB": xgb.XGBClassifier(
        eval_metric='logloss',
        max_depth=3,
        learning_rate=0.1,
        n_estimators=100,
        random_state=SEED
    ),

    "SVM": SVC(probability=True, class_weight=class_weights),

    "LR": LogisticRegression(max_iter=500, class_weight=class_weights),

    "MLP": MLPClassifier(max_iter=500, random_state=SEED)
}

# ========= 9. CROSS VALIDATION =========
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

print("\n Running Classical Models...\n")

for name, model in models.items():
    pipe = build_pipeline(model)

    scores = cross_validate(
        pipe, X, y,
        cv=cv,
        scoring={
            'accuracy': 'accuracy',
            'precision': 'precision_macro',
            'recall': 'recall_macro',
            'f1': 'f1_macro'
        },
        error_score='raise'
    )

    results.append({
        "Model": name,
        "Type": "Classical",
        "Accuracy": np.mean(scores['test_accuracy']),
        "Precision": np.mean(scores['test_precision']),
        "Recall": np.mean(scores['test_recall']),
        "F1": np.mean(scores['test_f1'])
    })

# ========= 10. TRAIN TEST =========
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], stratify=y, random_state=SEED
)

# ========= 11. TUNING =========
print("\n🔍 Tuning XGB...")

xgb_pipe = build_pipeline(models["XGB"])

grid = GridSearchCV(
    xgb_pipe,
    {'model__n_estimators': [100, 150], 'model__max_depth': [3, 4]},
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

# ========= 12. FINAL EVALUATION =========
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\n📊 CLASSIFICATION REPORT\n")
print(classification_report(y_test, y_pred))

results.append({
    "Model": "XGB_TUNED",
    "Type": "Final",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_proba)
})

# ========= 13. SAVE =========
joblib.dump(best_model, "final_model.pkl")

# ========= 14. QUANTUM MODEL =========
print("\n⚛️ Running VQC...")

if len(num_cols) >= 4:

    X_small = X[num_cols].iloc[:, :4].values   # already numpy

    Xtr_q, Xte_q, ytr_q, yte_q = train_test_split(
        X_small, y,
        test_size=CONFIG['test_size'],
        stratify=y,
        random_state=SEED
    )

    # 🔥 CRITICAL FIX (FORCE NUMPY + SHAPE)
    ytr_q = np.array(ytr_q).ravel()
    yte_q = np.array(yte_q).ravel()

    # (extra safe)
    Xtr_q = np.array(Xtr_q)
    Xte_q = np.array(Xte_q)

    # Scale for quantum
    scaler = MinMaxScaler(feature_range=(0, 2*np.pi))
    Xtr_q = scaler.fit_transform(Xtr_q)
    Xte_q = scaler.transform(Xte_q)

    # Quantum circuit
    feature_map = ZZFeatureMap(feature_dimension=Xtr_q.shape[1])
    ansatz = RealAmplitudes(num_qubits=Xtr_q.shape[1])

    # Model
    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=50)
    )

    # Train
    vqc.fit(Xtr_q, ytr_q)

    # Predict
    y_pred_q = vqc.predict(Xte_q)

    results.append({
        "Model": "VQC",
        "Type": "Quantum",
        "Accuracy": accuracy_score(yte_q, y_pred_q),
        "Precision": precision_score(yte_q, y_pred_q, zero_division=0),
        "Recall": recall_score(yte_q, y_pred_q, zero_division=0),
        "F1": f1_score(yte_q, y_pred_q, zero_division=0)
    })

else:
    print("⚠️ Not enough numeric features for VQC")

# ========= 15. RESULTS =========
results_df = pd.DataFrame(results)
print("\n🏁 FINAL RESULTS\n")
print(results_df)

results_df.to_csv("results.csv", index=False)

results_df.to_csv("/content/results.csv", index=False)
joblib.dump(best_model, "/content/final_model.pkl")
from google.colab import files
files.download("/content/results.csv")
files.download("/content/final_model.pkl")

Columns: ['college_id', 'student_name', 'gender', 'academic_year', 'branch', 'overall_cgpa', 'internships', 'placement_status', 'projects', 'crt_classes', 'attendance', 'backlogs']
Final dataset size: (506, 11)

🚀 Running Classical Models...


🔍 Tuning XGB...



📊 CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        72
           1       1.00      1.00      1.00        30

    accuracy                           1.00       102
   macro avg       1.00      1.00      1.00       102
weighted avg       1.00      1.00      1.00       102


⚛️ Running VQC...

🏁 FINAL RESULTS

       Model       Type  Accuracy  Precision    Recall        F1  ROC_AUC
0         RF  Classical  0.978237   0.969534  0.980775  0.974385      NaN
1        XGB  Classical  0.994059   0.995946  0.990000  0.992668      NaN
2        SVM  Classical  0.820248   0.805435  0.862737  0.808401      NaN
3         LR  Classical  0.895282   0.868586  0.902298  0.880545      NaN
4        MLP  Classical  0.885343   0.884225  0.834699  0.852484      NaN
5  XGB_TUNED      Final  1.000000   1.000000  1.000000  1.000000      1.0
6        VQC    Quantum  0.539216   0.242424  0.266667  0.253968      NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>